# Contrôles de la cible : les chiffres des hypothèses, recalculés

**But** : recalculer, dans un notebook, les chiffres de `docs/hypotheses_et_conclusions.md` marqués *(contrôle rapide)* : ils avaient été calculés à la volée pendant une discussion. Une fois ce notebook exécuté, ces chiffres peuvent être cités en renvoyant ici.

**Ce notebook est descriptif** : aucun modèle n'est entraîné ; il lit les jeux de données (`data/creditcard_pret_ingestion.csv` pour le niveau 0, `data/cleaned5_creditcard.csv` pour le niveau 5, `data/ML/train_dataset.csv`) et les fichiers enregistrés par les notebooks (probabilités hors pli de `ml_14`, décisions de l'essai à un seul modèle du comparatif global). Il ne choisit rien : il mesure.

Convention temporelle (`CLAUDE.md` du projet, rappelée dans les notebooks) : `PAY_1` porte sur le règlement de la facture d'août `BILL_AMT2` ; `PAY_AMT1` est le paiement de septembre.

**Contrôles** :
1. ne rien devoir protège-t-il du défaut ? (H3)
2. au contentieux, payer change-t-il le défaut ? (H3)
3. une codification de retard posée sur une dette nulle annonce-t-elle le défaut ? (H3)
4. un petit groupe démographique peu risqué : `EDUCATION` = 4 (H8)
5. avec un seul modèle et la règle « 1 pour 1 », quelle part des signalés est au contentieux ? (H5)
6. au rappel minimal de 60 %, que valent les derniers clients signalés ? et quel rappel garde une précision de 50 % ? (section 4 des hypothèses)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

pd.set_option("display.width", 250)
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
niveau_0 = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
niveau_5 = pd.read_csv(root_dir / "data" / "cleaned5_creditcard.csv")
train_ml = pd.read_csv(root_dir / "data" / "ML" / "train_dataset.csv")
au_ctx = (niveau_5["FLAG_CTX"] == 1) & (niveau_5["MOIS_SORTIE_CTX"] == -1)   # contentieux à M, définition 2
contentieux = niveau_5[au_ctx]

def ligne(libelle, d):
    return {"Groupe": libelle, "Clients": len(d), "Défauts": int(d["dpnm"].sum()),
            "Taux de défaut (%)": round(d["dpnm"].mean() * 100, 1) if len(d) else np.nan}

def montrer(lignes, titre):
    display(Markdown(f"**{titre}**"))
    display(pd.DataFrame(lignes).set_index("Groupe"))

## 1. Ne rien devoir protège-t-il du défaut ?

In [2]:
factures = [f"BILL_AMT{n}" for n in range(1, 7)]
montrer([
    ligne("Tous les clients (niveau 5)", niveau_5),
    ligne("Sans dette en septembre (BILL_AMT1 ≤ 0)", niveau_5[niveau_5["BILL_AMT1"] <= 0]),
    ligne("Sans aucune dette sur les six mois (toutes les factures ≤ 0)", niveau_5[(niveau_5[factures] <= 0).all(axis=1)]),
], "Niveau 5 : taux de défaut selon la dette")

**Niveau 5 : taux de défaut selon la dette**

,Clients,Défauts,Taux de défaut (%)
Groupe,,,
Tous les clients (niveau 5),28851,6272,21.7
Sans dette en septembre (BILL_AMT1 ≤ 0),1649,300,18.2
Sans aucune dette sur les six mois (toutes les factures ≤ 0),25,6,24.0


**Comment lire** : si les clients sans dette font défaut presque aussi souvent que la moyenne, ne rien devoir ne protège pas du défaut tel qu'il est codé : un indice que la cible n'est pas (seulement) un défaut de paiement observé.

## 2. Au contentieux, payer change-t-il le défaut ?

In [3]:
montrer([
    ligne("Contentieux à M, tous", contentieux),
    ligne("A payé en août et en septembre (PAY_AMT2 > 0 et PAY_AMT1 > 0)", contentieux[(contentieux["PAY_AMT1"] > 0) & (contentieux["PAY_AMT2"] > 0)]),
    ligne("N'a payé ni en août ni en septembre", contentieux[(contentieux["PAY_AMT1"] == 0) & (contentieux["PAY_AMT2"] == 0)]),
    ligne("Facture de septembre nulle ou négative", contentieux[contentieux["BILL_AMT1"] <= 0]),
    ligne("A réglé en septembre au moins la facture d'août (PAY_AMT1 ≥ BILL_AMT2 > 0)", contentieux[(contentieux["BILL_AMT2"] > 0) & (contentieux["PAY_AMT1"] >= contentieux["BILL_AMT2"])]),
], "Contentieux à M (niveau 5) : taux de défaut selon les paiements")

**Contentieux à M (niveau 5) : taux de défaut selon les paiements**

,Clients,Défauts,Taux de défaut (%)
Groupe,,,
"Contentieux à M, tous",3004,2117,70.5
A payé en août et en septembre (PAY_AMT2 > 0 et PAY_AMT1 > 0),1617,1143,70.7
N'a payé ni en août ni en septembre,420,282,67.1
Facture de septembre nulle ou négative,0,0,NaN
A réglé en septembre au moins la facture d'août (PAY_AMT1 ≥ BILL_AMT2 > 0),0,0,NaN


**Comment lire** : si les clients du contentieux qui paient font défaut autant que les autres, les paiements ne permettent pas de les sortir du contentieux.

## 3. Une codification de retard posée sur une dette nulle annonce-t-elle le défaut ?

In [4]:
montrer([
    ligne("Niveau 0 : PAY_1 ≥ 2, tous", niveau_0[niveau_0["PAY_1"] >= 2]),
    ligne("Niveau 0 : PAY_1 ≥ 2 et facture d'août nulle ou négative (BILL_AMT2 ≤ 0)", niveau_0[(niveau_0["PAY_1"] >= 2) & (niveau_0["BILL_AMT2"] <= 0)]),
    ligne("Niveau 5 : faux retard neutralisé (FAUX_CODAGE = 1)", niveau_5[niveau_5["FAUX_CODAGE"] == 1]),
    ligne("Niveau 5 : retard sur compte endormi neutralisé (SURVEILLANCE_RECENTE = 1)", niveau_5[niveau_5["SURVEILLANCE_RECENTE"] == 1]),
    ligne("Niveau 5 : PAY_1 ≥ 2 et facture d'août nulle ou négative, après correction", niveau_5[(niveau_5["PAY_1"] >= 2) & (niveau_5["BILL_AMT2"] <= 0)]),
], "Codifications de retard posées sur une dette nulle")

**Codifications de retard posées sur une dette nulle**

,Clients,Défauts,Taux de défaut (%)
Groupe,,,
"Niveau 0 : PAY_1 ≥ 2, tous",3130,2177,69.6
Niveau 0 : PAY_1 ≥ 2 et facture d'août nulle ou négative (BILL_AMT2 ≤ 0),63,40,63.5
Niveau 5 : faux retard neutralisé (FAUX_CODAGE = 1),45,15,33.3
Niveau 5 : retard sur compte endormi neutralisé (SURVEILLANCE_RECENTE = 1),179,90,50.3
"Niveau 5 : PAY_1 ≥ 2 et facture d'août nulle ou négative, après correction",11,8,72.7


**Comment lire** : si ces clients font défaut bien plus que la moyenne, la codification de la banque annonce le défaut même sans dette : la cible suit plutôt l'étiquette de la banque. Les effectifs sont petits : ordres de grandeur.

## 4. Un petit groupe peu risqué : `EDUCATION` = 4 (train du ML)

In [5]:
t = train_ml.groupby("EDUCATION")["dpnm"].agg(Clients="size", Défauts="sum")
t["Taux de défaut (%)"] = (t["Défauts"] / t["Clients"] * 100).round(1)
t["Part du train (%)"] = (t["Clients"] / t["Clients"].sum() * 100).round(2)
display(t)

,Clients,Défauts,Taux de défaut (%),Part du train (%)
EDUCATION,,,,
1,6771,919,13.6,34.98
2,9154,1565,17.1,47.29
3,3127,580,18.5,16.15
4,306,20,6.5,1.58


## 5. Un seul modèle, règle « 1 pour 1 » : quelle part des signalés est au contentieux ?

Décisions hors pli enregistrées par `comparatif_global/essai_jeu_de_base.ipynb` (meilleur modèle de la variante complète, sur la précision moyenne), croisées avec le contentieux à M (définition 2, connu pour les clients du niveau 5).

In [6]:
from sklearn.metrics import average_precision_score
fichier = root_dir / "data" / "ML" / "comparatif_global" / "jeu_de_base.csv"
if not fichier.exists():
    display(Markdown("⚠️ `jeu_de_base.csv` absent : lancer `essai_jeu_de_base.ipynb` avec sa cellule d'enregistrement."))
else:
    un = pd.read_csv(fichier)
    noms = sorted({c.split(" | ")[2] for c in un.columns if c.startswith("proba | Complet |")})
    nom = max(noms, key=lambda n: average_precision_score(un["dpnm"], un[f"proba | Complet | {n}"]))
    d = un.merge(niveau_5[["ID"]].assign(contentieux=au_ctx.to_numpy().astype(int)), on="ID", how="left")
    d["statut"] = d["contentieux"].map({1: "Contentieux à M", 0: "Hors contentieux"}).fillna("Retiré au nettoyage")
    d["signale"] = d[f"decision | Complet | {nom}"] == 1
    t = d.groupby("statut").agg(Clients=("ID", "size"), Défauts=("dpnm", "sum"), Signalés=("signale", "sum"))
    t["Défauts détectés"] = d[d["signale"]].groupby("statut")["dpnm"].sum()
    t["Part des signalés (%)"] = (t["Signalés"] / t["Signalés"].sum() * 100).round(1)
    t["Défauts du groupe détectés (%)"] = (t["Défauts détectés"] / t["Défauts"] * 100).round(1)
    display(Markdown(f"**Jeu de base, un seul modèle ({nom}), règle « 1 pour 1 »** : {int(d['signale'].sum())} clients signalés"))
    display(t)

**Jeu de base, un seul modèle (CatBoost), règle « 1 pour 1 »** : 3709 clients signalés

,Clients,Défauts,Signalés,Défauts détectés,Part des signalés (%),Défauts du groupe détectés (%)
statut,,,,,,
Contentieux à M,3004,2117,2890,2053,77.9,97.0
Hors contentieux,25847,4155,808,411,21.8,9.9
Retiré au nettoyage,1149,364,11,8,0.3,2.2


## 6. Au rappel minimal de 60 % : la marge, et le rappel possible à 50 % de précision

Probabilités hors pli de CatBoost dans `ml_14` (train du ML, sans contentieux).

In [7]:
p = pd.read_csv(root_dir / "data" / "ML" / "probas_hors_pli" / "ml_14_v8.csv")
y, s = p["dpnm"].to_numpy(), p["CatBoost"].to_numpy()
ordre = np.argsort(-s, kind="mergesort")
cumul = np.cumsum(y[ordre])
n = np.arange(1, len(y) + 1)
lignes = []
for r in (0.30, 0.40, 0.50, 0.60, 0.70, 0.80):
    k = int(np.searchsorted(cumul, np.ceil(r * y.sum()))) + 1
    marge = y[ordre][k - int(0.1 * k):k].mean()
    lignes.append({"Rappel": f"{r:.0%}", "Précision (%)": round(cumul[k - 1] / k * 100, 1), "Clients signalés (%)": round(k / len(y) * 100, 1),
                   "Fausses alertes par défaut détecté": round((k - cumul[k - 1]) / cumul[k - 1], 1),
                   "Précision marginale, 10 % derniers signalés (%)": round(marge * 100, 1)})
display(Markdown(f"**CatBoost de ml_14, probabilités hors pli** (taux de défaut {y.mean() * 100:.1f} %)"))
display(pd.DataFrame(lignes).set_index("Rappel"))
for plancher in (0.50, 0.40):
    possibles = np.where((cumul / n >= plancher) & (n >= 50))[0]
    k = possibles.max() + 1
    display(Markdown(f"Précision d'au moins {plancher:.0%} : rappel maximal **{cumul[k - 1] / y.sum() * 100:.1f} %** ({k} clients signalés)"))

**CatBoost de ml_14, probabilités hors pli** (taux de défaut 15.9 %)

,Précision (%),Clients signalés (%),Fausses alertes par défaut détecté,"Précision marginale, 10 % derniers signalés (%)"
Rappel,,,,
30%,41.2,11.6,1.4,25.9
40%,35.6,17.9,1.8,20.2
50%,30.5,26.1,2.3,17.2
60%,27.3,35.0,2.7,18.5
70%,24.0,46.4,3.2,14.8
80%,21.8,58.5,3.6,12.0


Précision d'au moins 50% : rappel maximal **12.0 %** (742 clients signalés)

Précision d'au moins 40% : rappel maximal **31.7 %** (2442 clients signalés)

**Suite** : une fois ce notebook exécuté, remplacer dans `docs/hypotheses_et_conclusions.md` la mention *(contrôle rapide)* par un renvoi à ce notebook, pour chaque chiffre retrouvé.